# XTNeighbor Repertoire Comparison Benchmark against Compairr

This notebook aims to provide a reproducible benchmark of XTNeighbor on repertoire comparison task against the state-of-the-art tool named Compairr ([Rognes et al.](https://doi.org/10.1093/bioinformatics/btac505)). That is, given a number of immune repertoires and a Hamming distance threshold, the algorithm computes the overlap of each pairwise repertoires. The overlap of two repertoires is defined as the count of each similar sequences pair (according to the input threshold) found across these repertoires. The dataset is obtained from [Emerson et al](https://doi.org/10.1038/ng.3822).

The notebook is divided into 6 steps as follow:
1. __Configuration:__ select the number of experiment repeats and maximum dataset size. Note that the largest option of dataset size requires high-RAM VM which requires paid Google Colab account.
2. __Benchmark Setup:__ install dependencies and implementations of various algorithms and compile them if need be.
3. __Repertoire Comparison Benchmark:__ perform benchmark comparing Compairr on CPU, SymDel algorithm on CPU and XTNeighbor on GPU at threshold `d=1,2`.
4. __Result Download:__ download the benchmark measurement as csv file.

Warning: some sections take up to 1 hour to run. The run time is remarked at each section's heading.

More information can be found in our [preprint paper](https://doi.org/10.48550/arXiv.2403.09010) and our [Github repository](https://github.com/heartnetkung/XT-neighbor).

## 0. Configuration

In [ ]:
# @title Configure the runtime and number of experiment repeats. High RAM is only available in Colab's premium plan.
n_repeat = 1 # @param ["1", "10", "30"] {type:"raw"}
high_ram = True # @param {type:"boolean"}

## 1. Benchmark Setup (run time ~ 3 min)

install dependency

In [2]:
! pip install -q pyrepseq

In [ ]:
import os.path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import time
import re
import random
import pyrepseq
import symscan
try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

clone the projects

In [4]:
if not os.path.exists("compairr"):
    !git clone https://github.com/uio-bmi/compairr.git

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

compile XTNeighbor-streaming

In [5]:
! mkdir -p {repo_path}xtneighbor_streaming/build
! cd {repo_path}xtneighbor_streaming/build; cmake ..;make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/xtneighbor_streaming/build
[ 50%] Linking CUDA executable xt_neighbor
[100%] Built target xt_neighbor


compile Compairr

In [6]:
!cd compairr; make

make -C src compairr
make[1]: Entering directory '/home/andreas/repos/XT-neighbor/benchmarks/compairr/src'
make[1]: 'compairr' is up to date.
make[1]: Leaving directory '/home/andreas/repos/XT-neighbor/benchmarks/compairr/src'


prepare repertoire info

In [7]:
def read_info():
  ans = pd.read_csv(f'{repo_path}data/info.csv')
  end = np.cumsum(ans['count'])
  ans['start'] = np.concatenate(([0],end[:-1]))
  ans['end'] = end
  return ans

info = read_info()
info

,file,count,start,end
0,HIP09789.tsv.gz,277587,0,277587
1,HIP13497.tsv.gz,166649,277587,444236
2,HIP08076.tsv.gz,257508,444236,701744
3,HIP13932.tsv.gz,180216,701744,881960
4,HIP09001.tsv.gz,299802,881960,1181762
...,...,...,...,...
585,HIP14211.tsv.gz,177784,107499856,107677640
586,HIP05574.tsv.gz,220719,107677640,107898359
587,HIP13933.tsv.gz,103325,107898359,108001684
588,HIP13414.tsv.gz,142422,108001684,108144106


prepare input data

In [8]:
N_FILES=5

def read_input():
  ! mkdir -p emerson_data
  for i in range(1,N_FILES+1):
    ! unzip -n {repo_path}data/emerson_rep"$i".zip -d emerson_data
  reps = []
  for i in range(1,N_FILES+1):
    reps.append(pd.read_csv(f'emerson_data/emerson_rep{i}.txt'))
  return pd.concat(reps,ignore_index=True)

data = read_input()
print(data.head())

Archive:  ../data/emerson_rep1.zip
Archive:  ../data/emerson_rep2.zip
Archive:  ../data/emerson_rep3.zip
Archive:  ../data/emerson_rep4.zip
Archive:  ../data/emerson_rep5.zip
                 cdr3  count
0        CASSLDSYEQYF     25
1         CASSEAYEQYF      8
2  CASSLGQGRTSGHYEQYF      4
3       CASLGQLNTEAFF      2
4     CASSLPAGDTGELFF  16951


## 2. Repertoire Comparison Benchmark (run time ~ 30 min at n_repeat=1,high_ram=False and ~ 120 min at n_repeat=1,high_ram=True)

check GPU availability

In [9]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

input preparation code

In [10]:
def sample_repertoire(data,info,n,random_state=0):
  corrupted_files = ['HIP14092.tsv.gz','HIP04958.tsv.gz']
  info_subset = info[~info['file'].isin(corrupted_files)][:220-len(corrupted_files)].sample(n, random_state=random_state)
  reps = []
  rep_col = []
  for i in range(len(info_subset)):
    row = info_subset.iloc[i,:]
    reps.append(data[row['start']:row['end']])
    rep_col += [i]*row['count']
  ans = pd.concat(reps,ignore_index=True)
  ans.rename(columns={'count':'duplicate_count','cdr3':'cdr3_aa'},inplace=True)
  ans['repertoire_id'] = rep_col
  return ans, info_subset

def prepare(reps,info_subset):
  reps.to_csv('compairr_input1.txt',index=False,sep='\t')
  reps.to_csv('xt_input1.txt',index=False,columns=['cdr3_aa','duplicate_count'])
  info_subset['count'].to_csv('xt_input2.txt',index=False)
  return reps['cdr3_aa'].tolist(), reps['duplicate_count'].tolist(), info_subset['count'].tolist()

SymDel algorithm implementation

In [ ]:
def unique_dict(seqs):
  ans = {}
  for i in range(len(seqs)):
    key = seqs[i]
    if ans.get(key) is None:
      ans[key] = [i]
    else:
      ans[key].append(i)
  return ans, list(ans.keys())

def _idx_to_repoverlap(idx, useqs, indexMap=None, dup_counts=None, rep_sizes=None):
    dup_counts = np.asarray(dup_counts, dtype=np.int64)
    rep_sizes = np.asarray(rep_sizes)
    n_rep, n_useq, n_seq = len(rep_sizes), len(useqs), len(dup_counts)

    rep_ids = np.repeat(np.arange(n_rep), rep_sizes)
    useq_ids = np.empty(n_seq, dtype=np.int64)
    for u, positions in enumerate(indexMap.values()):
        useq_ids[positions] = u

    rep_profile = csr_matrix((dup_counts, (useq_ids, rep_ids)), shape=(n_useq, n_rep))

    if len(idx):
        arr = np.asarray(idx)                # (i, j, d) triples
        i_arr, j_arr = arr[:, 0], arr[:, 1]
        cross = rep_profile[i_arr].T @ rep_profile[j_arr]
    else:
        cross = 0

    # the "sequence overlaps itself" diagonal term
    self_term = rep_profile.T @ rep_profile   
    return np.asarray((cross + self_term).todense())

def symdel_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_sizes=None):
  start = time.time()
  measure = 'hamming' if is_hamming else None
  indexMap, useqs = unique_dict(seqs)
  if is_hamming:
    raw_output = pyrepseq.symdel(useqs, max_edits=distance,
                               custom_distance=measure, max_custom_distance=distance)
  else:
    raw_output = pyrepseq.symdel(useqs, max_edits=distance)
  end1 = time.time()

  ans = _idx_to_repoverlap(raw_output, useqs, indexMap=indexMap, dup_counts=dup_counts, rep_sizes=rep_sizes)

  end3 = time.time()
  return ans


def symscan_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_sizes=None):
  start = time.time()
  indexMap, useqs = unique_dict(seqs)

  if is_hamming:
    raw_output = symscan.get_neighbors_within(useqs, max_distance=distance, distance_type='hamming')
  else:
    raw_output = symscan.get_neighbors_within(useqs, max_distance=distance)
  raw_output = list(zip(*raw_output))
  end1 = time.time()

  ans = _idx_to_repoverlap(raw_output, useqs, indexMap=indexMap, dup_counts=dup_counts, rep_sizes=rep_sizes)
  
  end2 = time.time()
  print(f"symscan: {end1-start:.2f} {end2-end1:.2f}")
  return ans

standardize all algorithms to the same API

In [12]:
def xt_neighbor_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_sizes=None):
  n,N = len(seqs), len(rep_sizes)
  if is_hamming:
    ! {repo_path}xtneighbor_streaming/build/xt_neighbor -i "xt_input1.txt" -n "$n" -I "xt_input2.txt" -N "$N" -d "$distance" -m "hamming" -o "xt_output.txt"
  else:
    ! {repo_path}xtneighbor_streaming/build/xt_neighbor -i "xt_input1.txt" -n "$n" -I "xt_input2.txt" -N "$N" -d "$distance" -o "xt_output.txt"

def compairr_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_sizes=None):
  if is_hamming:
    !./compairr/src/compairr -g -a -m compairr_input1.txt compairr_input1.txt -d "$distance" --cdr3 -o output.tsv  > /dev/null 2>&1
  else:
    !./compairr/src/compairr -g -a -m compairr_input1.txt compairr_input1.txt -d "$distance" --cdr3 -o output.tsv -i  > /dev/null 2>&1

benchmarking code

In [13]:
sizes = [1,2,4,8,16,32,64]
algorithms = {
    'symdel':symdel_overlap,
    'symscan':symscan_overlap,
    'xt_streaming': xt_neighbor_overlap,
    'compairr': compairr_overlap
}
limits = {
    'symdel_1':16,
    'symdel_2':16,
    'symscan_1':64,
    'symscan_2':32,
    'xt_streaming_1':64,
    'xt_streaming_2':32,
    'compairr_1':64,
    'compairr_2':16,
}
if not high_ram:
    sizes = [1,2,4,8]

result_data = {'runtime':[],'algorithm':[],'n_sequence':[],'distance':[],'measure':[],'n_repertoire':[]}

def run_exp(distance, is_hamming):
    for i in range(n_repeat):
        for size in sizes:
            seq_info, reps = sample_repertoire(data,info,size,random_state=i)
            seqs, dup_counts, rep_sizes = prepare(seq_info,reps)
            _len = len(seqs)
            for alg_name in algorithms:
                limit = limits.get(f"{alg_name}_{distance}")
                if limit is not None and limit <size:
                    continue

                # perform
                start = time.time()
                algorithms[alg_name](distance,is_hamming,seqs, dup_counts, rep_sizes)
                end = time.time()

                # record
                print(f'{size:,}',_len,alg_name,i,round((end-start)*100)/100)
                result_data['runtime'].append(end-start)
                result_data['algorithm'].append(alg_name)
                result_data['n_sequence'].append(len(seqs))
                result_data['distance'].append(distance)
                result_data['measure'].append('hamming' if is_hamming else 'leven')
                result_data['n_repertoire'].append(size)

In [14]:
run_exp(distance=1, is_hamming=False)

1 241817 symdel 0 3.68
symscan: 0.21 0.21
1 241817 symscan 0 0.44
1 241817 xt_streaming 0 0.62
1 241817 compairr 0 0.6
2 407713 symdel 0 6.35
symscan: 0.25 0.39
2 407713 symscan 0 0.68
2 407713 xt_streaming 0 0.72
2 407713 compairr 0 0.97
4 765218 symdel 0 11.25
symscan: 0.56 0.91
4 765218 symscan 0 1.55
4 765218 xt_streaming 0 0.87
4 765218 compairr 0 2.46
8 1601656 symdel 0 25.81
symscan: 2.26 1.74
8 1601656 symscan 0 4.17
8 1601656 xt_streaming 0 1.29
8 1601656 compairr 0 6.76


In [15]:
run_exp(distance=1, is_hamming=True)

1 241817 symdel 0 3.41
symscan: 0.10 0.18
1 241817 symscan 0 0.29
1 241817 xt_streaming 0 0.6
1 241817 compairr 0 0.4
2 407713 symdel 0 5.39
symscan: 0.19 0.28
2 407713 symscan 0 0.49
2 407713 xt_streaming 0 0.67
2 407713 compairr 0 0.61
4 765218 symdel 0 10.92
symscan: 0.56 0.61
4 765218 symscan 0 1.24
4 765218 xt_streaming 0 0.83
4 765218 compairr 0 1.31
8 1601656 symdel 0 23.44
symscan: 1.28 1.39
8 1601656 symscan 0 2.81
8 1601656 xt_streaming 0 1.2
8 1601656 compairr 0 4.18


In [ ]:
run_exp(distance=2, is_hamming=True)

1 241817 symdel 0 24.54
symscan: 0.72 0.88
1 241817 symscan 0 1.69
1 241817 xt_streaming 0 1.2
1 241817 compairr 0 20.12
2 407713 symdel 0 41.94
symscan: 1.58 2.03
2 407713 symscan 0 3.83
2 407713 xt_streaming 0 1.79
2 407713 compairr 0 36.39
4 765218 symdel 0 90.66
symscan: 3.75 5.05
4 765218 symscan 0 9.23
4 765218 xt_streaming 0 3.21
4 765218 compairr 0 76.88


## 3. Result Download (run time < 1 min)

In [ ]:
pd.DataFrame(result_data).to_csv('compairr_benchmark.csv')
if colab:
    files.download('compairr_benchmark.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>